# House Prices：变量选择与工程的艺术
## 基于 Kaggle House Prices 竞赛的完整特征工程方法论

> 79 个特征 → 系统筛选 → 编码对比 → 模型集成。每一步都有数据支撑的决策。

## 第 1 章 · 问题定义

### 1.1 竞赛背景

House Prices: Advanced Regression Techniques 是 Kaggle 上最经典的回归竞赛之一。数据来自爱荷华州 Ames 市 2006-2010 年的住宅销售记录，79 个解释变量覆盖了从地基到屋顶、从室内到室外的几乎所有建筑属性。

跟前两个项目（Titanic 的二分类、Bike Sharing 的时间序列）不同，这个项目面对的是**大规模混合型表格数据**——43 个类别特征 + 36 个数值特征，NA 在不同列代表完全不同的语义（"没有地下室" vs "没有壁炉" vs "线数据丢失"）。这才是真实业务中最常见的场景。

### 1.2 评估指标：RMSLE

跟 Bike Sharing 一样，House Prices 用的是 RMSLE（官方叫 RMSE between log）：

$$RMSLE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(\log(\hat{y_i}+1) - \log(y_i+1))^2}$$

RMSLE 的核心特性：
- **惩罚比例误差**：预测 10 万实际 15 万（误差 50%）vs 预测 35 万实际 40 万（误差 14%），虽然绝对值差一样，但 RMSLE 对前者惩罚更重
- **对高价不敏感**：Ames 的 75 分位房价大概是 21 万，99 分位约 61 万——长尾上的少数豪宅不会主导评估

目标：RMSLE < 0.12，对应 Kaggle top 20%。

### 1.3 导入库

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 中文字体配置
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 建模库
from sklearn.model_selection import KFold, cross_val_score, RepeatedKFold
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.feature_selection import mutual_info_regression, RFECV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.inspection import partial_dependence, PartialDependenceDisplay
from sklearn.model_selection import cross_validate
import category_encoders as ce
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import optuna

# 版本确认
print(f'Python:    {pd.__version__}')
print(f'numpy:     {np.__version__}')
print(f'pandas:    {pd.__version__}')
import sklearn; print(f'sklearn:   {sklearn.__version__}')
print(f'xgboost:   {xgb.__version__}' if hasattr(xgboost, '__version__') else xgboost.VERSION)
print(f'lightgbm:  {lgb.__version__}' if hasattr(lightgbm, '__version__') else lightgbm.VERSION)
print(f'catboost:  {catboost.__version__}')
print(f'optuna:    {optuna.__version__}')
print(f'category_encoders: {ce.__version__}')

In [ ]:
# 加载数据
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f'训练集: {train.shape}')
print(f'测试集: {test.shape}')
print(f'\n训练集列数: {train.shape[1]} (79 特征 + Id + SalePrice = 81)')
print(f'测试集列数: {test.shape[1]} (79 特征 + Id = 80)')

# 保存 Id 用于最终提交
test_id = test['Id']
train.drop('Id', axis=1, inplace=True)
test.drop('Id', axis=1, inplace=True)

### 1.4 特征概览

79 个特征按类型分组：

| 类别 | 数量 | 示例 |
|------|------|------|
| 面积类 | ~15 | GrLivArea, TotalBsmtSF, LotArea, GarageArea, WoodDeckSF... |
| 质量/评级类 | ~10 | OverallQual, ExterQual, KitchenQual, BsmtQual... |
| 年份/时间类 | ~4 | YearBuilt, YearRemodAdd, YrSold, MoSold |
| 设施有无类 | ~20 | FireplaceQu, PoolQC, Fence, GarageType, BsmtFinType1... |
| 位置/分类类 | ~20 | Neighborhood, MSZoning, LotConfig, HouseStyle, SaleType... |
| 其他数值 | ~10 | Fireplaces, FullBath, Bedroom, Kitchen, TotRmsAbvGrd... |

结构上这是一个典型的房地产评估数据集——从地块属性到建筑质量到室内设施，覆盖了影响房价的主要维度。

## 第 2 章 · EDA —— 数据全景与缺失值语义

### 2.1 目标变量：SalePrice

In [ ]:
# === SalePrice 分布分析 ===
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 原始分布
axes[0].hist(train['SalePrice'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(train['SalePrice'].mean(), color='red', linestyle='--', label=f'Mean: {train["SalePrice"].mean():,.0f}')
axes[0].axvline(train['SalePrice'].median(), color='orange', linestyle='--', label=f'Median: {train["SalePrice"].median():,.0f}')
axes[0].set_title('SalePrice 原始分布', fontweight='normal')
axes[0].set_xlabel('SalePrice ($)')
axes[0].legend()

# log 变换后分布
log_sp = np.log1p(train['SalePrice'])
axes[1].hist(log_sp, bins=50, color='darkseagreen', edgecolor='white', alpha=0.85)
axes[1].axvline(log_sp.mean(), color='red', linestyle='--', label=f'Mean: {log_sp.mean():.3f}')
axes[1].axvline(log_sp.median(), color='orange', linestyle='--', label=f'Median: {log_sp.median():.3f}')
axes[1].set_title('log(SalePrice) 变换后', fontweight='normal')
axes[1].set_xlabel('log(SalePrice)')
axes[1].legend()

# QQ plot
from scipy import stats
stats.probplot(log_sp, dist="norm", plot=axes[2])
axes[2].set_title('QQ Plot (log变换后)', fontweight='normal')

plt.tight_layout()
plt.show()

print(f'SalePrice 统计:')
print(f'  Mean:     {train["SalePrice"].mean():,.0f}')
print(f'  Median:   {train["SalePrice"].median():,.0f}')
print(f'  Std:      {train["SalePrice"].std():,.0f}')
print(f'  Skewness: {train["SalePrice"].skew():.3f}')
print(f'  Kurtosis: {train["SalePrice"].kurtosis():.3f}')
print(f'  Min:      {train["SalePrice"].min():,.0f}')
print(f'  Max:      {train["SalePrice"].max():,.0f}')
print(f'  Q(0.01):  {train["SalePrice"].quantile(0.01):,.0f}')
print(f'  Q(0.99):  {train["SalePrice"].quantile(0.99):,.0f}')

In [ ]:
# === 异常值检测 ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GrLivArea vs SalePrice（已知异常区）
axes[0].scatter(train['GrLivArea'], train['SalePrice'], alpha=0.4, s=15, color='steelblue')
axes[0].set_xlabel('GrLivArea (sqft)')
axes[0].set_ylabel('SalePrice ($)')
axes[0].set_title('GrLivArea vs SalePrice', fontweight='normal')

# 标注异常区域
axes[0].axvline(x=4000, color='red', linestyle='--', alpha=0.6, label='GrLivArea=4000')
axes[0].axhline(y=300000, color='orange', linestyle='--', alpha=0.6, label='SalePrice=300k')
axes[0].fill_between([4000, train['GrLivArea'].max()], 0, 300000, alpha=0.1, color='red')
axes[0].legend(fontsize=9)

# 标记异常点
anomalies = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)]
axes[0].scatter(anomalies['GrLivArea'], anomalies['SalePrice'], color='red', s=50, zorder=5)
for _, row in anomalies.iterrows():
    axes[0].annotate(f"${row['SalePrice']:,.0f}", 
                     (row['GrLivArea'], row['SalePrice']),
                     fontsize=8, color='red')

print(f'GrLivArea > 4000 且 SalePrice < 300k 的异常样本: {len(anomalies)}')
if len(anomalies) > 0:
    display(anomalies[['GrLivArea', 'TotalBsmtSF', 'OverallQual', 'YearBuilt', 'SalePrice']])

# SalePrice 箱线图（高价位区域）
axes[1].boxplot(train['SalePrice'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_ylabel('SalePrice ($)')
axes[1].set_title('SalePrice 箱线图', fontweight='normal')

# 标记极端值
q1, q3 = train['SalePrice'].quantile(0.25), train['SalePrice'].quantile(0.75)
iqr = q3 - q1
upper = q3 + 3 * iqr  # 3倍IQR（更保守）
extreme = train[train['SalePrice'] > upper]
print(f'\nSalePrice > Q3 + 3*IQR (${upper:,.0f}) 的极端值: {len(extreme)}')
if len(extreme) > 0:
    display(extreme[['GrLivArea', 'OverallQual', 'SalePrice']].sort_values('SalePrice', ascending=False).head(5))

plt.tight_layout()
plt.show()

### 2.2 数值特征全貌

In [ ]:
# === 数值特征与 SalePrice 的相关性 ===
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('SalePrice')

# Pearson + Spearman 相关系数
corr_pearson = train[numeric_cols + ['SalePrice']].corr()['SalePrice'].drop('SalePrice')
corr_spearman = train[numeric_cols + ['SalePrice']].corr(method='spearman')['SalePrice'].drop('SalePrice')

corr_df = pd.DataFrame({
    'Pearson': corr_pearson.abs().sort_values(ascending=False),
    'Spearman': corr_spearman.abs().sort_values(ascending=False)
}).sort_values('Pearson', ascending=False)

# Top 20
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

top20 = corr_df.head(20)
axes[0].barh(range(len(top20)), top20['Pearson'].values, color='steelblue', alpha=0.8)
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels(top20.index, fontsize=9)
axes[0].invert_yaxis()
axes[0].set_title('Top 20 数值特征 |Pearson| 相关系数', fontweight='normal')
axes[0].set_xlabel('|Pearson r|')
for i, v in enumerate(top20['Pearson'].values):
    axes[0].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=8)

# Bottom（弱相关特征）
bottom_n = corr_df.sort_values('Pearson').head(15)
axes[1].barh(range(len(bottom_n)), bottom_n['Pearson'].values, color='lightcoral', alpha=0.8)
axes[1].set_yticks(range(len(bottom_n)))
axes[1].set_yticklabels(bottom_n.index, fontsize=9)
axes[1].invert_yaxis()
axes[1].set_title('低相关数值特征（可能为噪声）', fontweight='normal')
axes[1].set_xlabel('|Pearson r|')
for i, v in enumerate(bottom_n['Pearson'].values):
    axes[1].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# === 面积类特征的共线性扫描 ===
area_cols = ['GrLivArea', '1stFlrSF', '2ndFlrSF', 'TotalBsmtSF', 
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
             'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'LotArea']

area_corr = train[area_cols].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(area_corr, dtype=bool), k=1)
sns.heatmap(area_corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=.5,
            cbar_kws={'shrink': 0.8})
plt.title('面积类特征相关矩阵（上三角）', fontweight='normal', fontsize=13)
plt.tight_layout()
plt.show()

# 验证已知共线关系
print('验证共线关系:')
print(f'  GrLivArea vs (1stFlrSF + 2ndFlrSF): 差值的绝对值 max = {abs(train["GrLivArea"] - (train["1stFlrSF"] + train["2ndFlrSF"])).max():.0f}')
print(f'  TotalBsmtSF vs (BsmtFinSF1 + BsmtFinSF2 + BsmtUnfSF): 差值的绝对值 max = {abs(train["TotalBsmtSF"] - (train["BsmtFinSF1"] + train["BsmtFinSF2"] + train["BsmtUnfSF"])).max():.0f}')

# 年份特征
year_cols = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']
print(f'\n年份特征:')
for yc in year_cols:
    corr_val = train[yc].corr(train['SalePrice'])
    print(f'  {yc}: r(SalePrice) = {corr_val:.3f}, NA count = {train[yc].isna().sum()}')

### 2.3 类别特征与 NA 语义分类

这是本项目区别于前两个项目的关键环节。House Prices 里有大量 NA，但它们的含义不是"数据丢了"，而是"这东西不存在"。我们需要逐个分析并给 NA 贴标签。

In [ ]:
# === 全量缺失值统计 ===
all_na = train.isna().sum()
all_na = all_na[all_na > 0].sort_values(ascending=False)

print('=== 训练集缺失值总览 ===')
for col, cnt in all_na.items():
    pct = cnt / len(train) * 100
    print(f'  {col:<25s} {cnt:5d}  ({pct:4.1f}%)')

print(f'\n有缺失值的特征: {len(all_na)} 个')
print(f'无缺失的特征: {79 - len(all_na)} 个')

In [ ]:
# === 类别特征的基数与 NA 分布 ===
cat_cols = train.select_dtypes(include=['object']).columns.tolist()

print('=== 类别特征基数与 NA 情况 ===')
print(f'{"特征":<22s} {"基数":>5s}  {"NA数":>6s}  {"NA%":>6s}  {"NA语义":<20s}')
print('-' * 75)

for col in cat_cols:
    cardinality = train[col].nunique()
    na_count = train[col].isna().sum()
    na_pct = na_count / len(train) * 100

    # NA 语义判定
    if na_count == 0:
        semantic = '(无NA)'
    elif col in ['Alley']:
        semantic = '无巷子通道'
    elif col.startswith('Bsmt'):
        semantic = '无地下室'
    elif col.startswith('Garage'):
        semantic = '无车库'
    elif col == 'FireplaceQu':
        semantic = '无壁炉'
    elif col == 'PoolQC':
        semantic = '无泳池'
    elif col == 'Fence':
        semantic = '无围栏'
    elif col == 'MiscFeature':
        semantic = '无杂项设施'
    elif col == 'MasVnrType':
        semantic = '无贴面(多数)/缺失'
    elif col == 'Electrical':
        semantic = '真缺失(仅1条)'
    else:
        semantic = '(检查)'

    print(f'  {col:<20s} {cardinality:5d}  {na_count:6d}  {na_pct:5.1f}%  {semantic:<20s}')

# 高基数类别特征
print('\n=== 高基数类别特征 (>= 10 类) ===')
high_card = [(col, train[col].nunique()) for col in cat_cols if train[col].nunique() >= 10]
for col, card in sorted(high_card, key=lambda x: -x[1]):
    na_cnt = train[col].isna().sum()
    print(f'  {col:<22s} 基数: {card:3d}  NA: {na_cnt:4d}')

In [ ]:
# === 数值特征的 NA 分布 ===
num_na = train.select_dtypes(include=[np.number]).isna().sum()
num_na = num_na[num_na > 0].sort_values(ascending=False)

print('=== 数值特征缺失值 ===')
for col, cnt in num_na.items():
    print(f'  {col:<22s} {cnt:5d}  ({cnt/len(train)*100:4.1f}%)  ', end='')
    if col == 'LotFrontage':
        print('真缺失 — 按 Neighborhood 中位数填充')
    elif col == 'MasVnrArea':
        print('无贴面→0 / 真缺失→0')
    elif col == 'GarageYrBlt':
        print('无车库→0（配合 has_garage flag）')
    else:
        print()

### 2.4 NA 语义分类表（完整版）

综合以上分析，训练集中 19 个有缺失值的特征可分为三类：

**第一类：结构性缺失**（NA = 不存在该设施）—— 填充为 "None" 或 "No_XXX" 标记

| 特征 | NA 数量 | NA 语义 | 填充值 |
|------|---------|---------|--------|
| Alley | 1369 (93.8%) | 无巷子通道 | "No_Alley" |
| BsmtQual | 37 (2.5%) | 无地下室 | "No_Basement" |
| BsmtCond | 37 (2.5%) | 无地下室 | "No_Basement" |
| BsmtExposure | 38 (2.6%) | 无地下室 | "No_Basement" |
| BsmtFinType1 | 37 (2.5%) | 无地下室 | "No_Basement" |
| BsmtFinType2 | 38 (2.6%) | 无地下室 | "No_Basement" |
| FireplaceQu | 690 (47.3%) | 无壁炉 | "No_Fireplace" |
| GarageType | 81 (5.5%) | 无车库 | "No_Garage" |
| GarageFinish | 81 (5.5%) | 无车库 | "No_Garage" |
| GarageQual | 81 (5.5%) | 无车库 | "No_Garage" |
| GarageCond | 81 (5.5%) | 无车库 | "No_Garage" |
| PoolQC | 1453 (99.5%) | 无泳池（极度稀疏） | "No_Pool" |
| Fence | 1179 (80.8%) | 无围栏 | "No_Fence" |
| MiscFeature | 1406 (96.3%) | 无杂项设施（极度稀疏） | "No_Misc" |

**第二类：混合型缺失**（部分为不存在，部分为真缺失）

| 特征 | NA 数量 | NA 语义 | 填充策略 |
|------|---------|---------|----------|
| MasVnrType | 8 (0.5%) | 多数无贴面（MasVnrArea=0），少数可能真缺失 | 填 "None"，后续通过 MasVnrArea 验证 |
| MasVnrArea | 8 (0.5%) | 无贴面则面积为 0 | 填 0 |
| GarageYrBlt | 81 (5.5%) | 无车库 → 建造年不适用 | 填 0，配合 has_garage flag |

**第三类：真缺失**（数据确实丢了）

| 特征 | NA 数量 | 填充策略 |
|------|---------|----------|
| LotFrontage | 259 (17.7%) | 按 Neighborhood 分组中位数填充 |
| Electrical | 1 (0.07%) | 填众数 |

这张表是第 3 章特征工程的地基——每个 NA 用什么策略填，不靠猜测。另外注意，PoolQC 和 MiscFeature 的 NA 占比超过 95%，我们后续会构造 has_pool / has_misc 的 binary flag 来保留"有/没有"这个信息，并考虑直接丢弃原始列。

In [ ]:
# === 2.5 异常值处理 ===
# 删除 GrLivArea > 4000 且 SalePrice < 300000 的异常样本
anomaly_mask = (train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)
anomaly_indices = train[anomaly_mask].index.tolist()

print(f'删除异常样本: {len(anomaly_indices)} 条')
print(f'  索引: {anomaly_indices}')

train_clean = train.drop(anomaly_indices).reset_index(drop=True)
print(f'  训练集: {train.shape[0]} → {train_clean.shape[0]} 条')

# 更新 train 引用（后续所有操作使用 train_clean）
train = train_clean

## 第 3 章 · 特征工程 —— 变量选择与工程的艺术

这是本项目的核心章节。79 个原始特征、复杂的 NA 语义、多样的编码策略——每一步都有对比实验支撑决策。

### 3.1 缺失值填充

In [ ]:
# === 合并 train + test 以便统一处理 ===
y = np.log1p(train['SalePrice'])  # 对数目标变量
all_data = pd.concat([train.drop('SalePrice', axis=1), test], axis=0, ignore_index=True)
print(f'合并数据: {all_data.shape} (train: {train.shape[0]}, test: {test.shape[0]})')

# === 第一类：结构性缺失 → 填字符串标记 ===
na_fill_str = {
    'Alley': 'No_Alley',
    'BsmtQual': 'No_Basement',
    'BsmtCond': 'No_Basement',
    'BsmtExposure': 'No_Basement',
    'BsmtFinType1': 'No_Basement',
    'BsmtFinType2': 'No_Basement',
    'FireplaceQu': 'No_Fireplace',
    'GarageType': 'No_Garage',
    'GarageFinish': 'No_Garage',
    'GarageQual': 'No_Garage',
    'GarageCond': 'No_Garage',
    'PoolQC': 'No_Pool',
    'Fence': 'No_Fence',
    'MiscFeature': 'No_Misc',
}
for col, fill_val in na_fill_str.items():
    all_data[col] = all_data[col].fillna(fill_val)

# === 第二类：混合型 → 区分处理 ===
all_data['MasVnrType'] = all_data['MasVnrType'].fillna('None')
all_data['MasVnrArea'] = all_data['MasVnrArea'].fillna(0)
all_data['GarageYrBlt'] = all_data['GarageYrBlt'].fillna(0)

# === 第三类：真缺失 ===
# LotFrontage — 按 Neighborhood 中位数填
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
all_data['LotFrontage'] = all_data['LotFrontage'].fillna(all_data['LotFrontage'].median())

# Electrical — 仅 1 条缺失，填众数
all_data['Electrical'] = all_data['Electrical'].fillna(all_data['Electrical'].mode()[0])

# === 测试集独有缺失（仅在 test 中出现 NA 的列） ===
extra_na_fill_mode = {
    'MSZoning': 'RL',       # Residential Low Density（最频）
    'Utilities': 'AllPub',  # 几乎所有样本都是 AllPub
    'Exterior1st': 'VinylSd',
    'Exterior2nd': 'VinylSd',
    'KitchenQual': 'TA',
    'Functional': 'Typ',
    'SaleType': 'WD',
}
for col, fill_val in extra_na_fill_mode.items():
    all_data[col] = all_data[col].fillna(fill_val)

# 地下/车库相关的数值缺失 → 填 0
extra_na_zero = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
                 'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']
for col in extra_na_zero:
    all_data[col] = all_data[col].fillna(0)

# === 构造 binary flag（"有/没有"信息显式化） ===
all_data['has_garage'] = (all_data['GarageType'] != 'No_Garage').astype(int)
all_data['has_basement'] = (all_data['BsmtQual'] != 'No_Basement').astype(int)
all_data['has_fireplace'] = (all_data['FireplaceQu'] != 'No_Fireplace').astype(int)
all_data['has_pool'] = (all_data['PoolQC'] != 'No_Pool').astype(int)
all_data['has_fence'] = (all_data['Fence'] != 'No_Fence').astype(int)
all_data['has_alley'] = (all_data['Alley'] != 'No_Alley').astype(int)

print(f'Binary flags: has_garage={all_data["has_garage"].sum()}, has_basement={all_data["has_basement"].sum()}')
print(f'  has_fireplace={all_data["has_fireplace"].sum()}, has_pool={all_data["has_pool"].sum()}')
print(f'  has_fence={all_data["has_fence"].sum()}, has_alley={all_data["has_alley"].sum()}')

# 验证无残留 NA
remaining_na = all_data.isna().sum().sum()
print(f'\n残留 NA: {remaining_na}')

### 3.2 新特征构造

In [ ]:
# === 基础衍生特征 ===
# 浴室总数
all_data['TotalBath'] = (all_data['FullBath'] + 0.5 * all_data['HalfBath'] +
                          all_data['BsmtFullBath'] + 0.5 * all_data['BsmtHalfBath'])

# 总面积
all_data['TotalSF'] = (all_data['GrLivArea'] + all_data['TotalBsmtSF'] +
                       all_data['WoodDeckSF'] + all_data['OpenPorchSF'])

# 房龄（以 2010 年为基准，因为 YrSold 最晚是 2010）
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']

# 各类 Porch 总面积
all_data['PorchSF'] = (all_data['OpenPorchSF'] + all_data['EnclosedPorch'] +
                       all_data['3SsnPorch'] + all_data['ScreenPorch'])

# 综合品质评分
all_data['OverallScore'] = all_data['OverallQual'] * all_data['OverallCond']

# === 高 ROI 特征（设计审核补充） ===
# 是否翻新过
all_data['WasRemodeled'] = (all_data['YearBuilt'] != all_data['YearRemodAdd']).astype(int)

# 车库年龄（无车库的强制填 0，靠 has_garage flag 区分）
all_data['GarageAge'] = all_data['YrSold'] - all_data['GarageYrBlt']
all_data.loc[all_data['has_garage'] == 0, 'GarageAge'] = 0

# 总面积的对数
all_data['LogTotalSF'] = np.log1p(all_data['TotalSF'])

# 品质 × 面积交互
all_data['Qual_x_SF'] = all_data['OverallQual'] * all_data['GrLivArea']

print(f'新增特征: TotalBath, TotalSF, HouseAge, RemodAge, PorchSF, OverallScore')
print(f'          WasRemodeled, GarageAge, LogTotalSF, Qual_x_SF')
print(f'总特征数: {all_data.shape[1]}')

### 3.3 类别编码对比实验

线性模型和树模型对编码方式的偏好不同——一个编码方案不能套所有模型。我们用 Ridge + LightGBM **双裁判**，对比四种编码策略的 CV 表现。

In [ ]:
# === 准备：分离回 train/test ===
train_len = len(train)
train_processed = all_data[:train_len].copy()
test_processed = all_data[train_len:].copy()

# === 类别特征列表 ===
cat_cols_updated = train_processed.select_dtypes(include=['object']).columns.tolist()

# === Ordinal 质量映射（Ex=5, Gd=4, TA=3, Fa=2, Po=1） ===
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0,
               'No_Basement': 0, 'No_Garage': 0, 'No_Fireplace': 0, 
               'No_Pool': 0, 'No_Fence': 0, 'No_Misc': 0, 'No_Alley': 0}

# 质量类特征（有序）
ordinal_quality_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                         'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual',
                         'GarageCond', 'PoolQC', 'Fence']

# 名义类特征（无序 → 标签编码）
nominal_cols = [c for c in cat_cols_updated if c not in ordinal_quality_cols]

def encode_ordinal(df, fit_df=None):
    """方案 A: 有序编码"""
    df = df.copy()
    for col in ordinal_quality_cols:
        if col in df.columns:
            df[col] = df[col].map(quality_map).fillna(0).astype(int)
    for col in nominal_cols:
        if col in df.columns:
            df[col] = pd.factorize(df[col])[0]
    return df

def encode_onehot(df, fit_df=None):
    """方案 B: One-Hot 编码"""
    df = df.copy()
    # 移除不参与 one-hot 的数值列
    numeric = df.select_dtypes(include=[np.number]).columns
    cat = df.select_dtypes(include=['object']).columns
    if fit_df is not None:
        # 用 fit_df 的列保证 train/test 一致
        cat_df = pd.get_dummies(df[cat], drop_first=True)
        # 只保留 fit 中存在的列
        fit_dummies = pd.get_dummies(fit_df[cat], drop_first=True)
        for c in fit_dummies.columns:
            if c not in cat_df.columns:
                cat_df[c] = 0
        cat_df = cat_df[fit_dummies.columns]
    else:
        cat_df = pd.get_dummies(df[cat], drop_first=True)
    result = pd.concat([df[numeric], cat_df], axis=1)
    return result

def encode_target(df, fit_df=None, fit_y=None):
    """方案 C: Target Encoding（用 category_encoders，OOF + smoothing）"""
    df = df.copy()
    numeric = df.select_dtypes(include=[np.number]).columns
    cat = df.select_dtypes(include=['object']).columns
    te = ce.TargetEncoder(cols=cat.tolist(), smoothing=20)
    if fit_df is not None and fit_y is not None:
        te.fit(fit_df[cat], fit_y)
        encoded = te.transform(df[cat])
    else:
        encoded = te.fit_transform(df[cat], fit_y)
    result = pd.concat([df[numeric], encoded], axis=1)
    return result

def encode_hybrid(df, fit_df=None, fit_y=None):
    """方案 D: 混合 — 质量类 ordinal + 名义类 target encoding"""
    df = df.copy()
    for col in ordinal_quality_cols:
        if col in df.columns:
            df[col] = df[col].map(quality_map).fillna(0).astype(int)
    numeric = df.select_dtypes(include=[np.number]).columns
    te = ce.TargetEncoder(cols=nominal_cols, smoothing=20)
    if fit_df is not None and fit_y is not None:
        te.fit(fit_df[nominal_cols], fit_y)
        encoded = te.transform(df[nominal_cols])
    else:
        encoded = te.fit_transform(df[nominal_cols], fit_y)
    result = pd.concat([df[numeric], encoded], axis=1)
    return result

print(f'类别特征: {len(cat_cols_updated)} 个')
print(f'  质量类(有序): {len(ordinal_quality_cols)} 个 — {ordinal_quality_cols}')
print(f'  名义类(无序): {len(nominal_cols)} 个 — 高基数: {[c for c in nominal_cols if train_processed[c].nunique() >= 10]}')

In [ ]:
# === 编码对比实验：Ridge + LightGBM 双裁判 ===
from sklearn.metrics import mean_squared_error

def rmsle_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

encodings = {
    'Ordinal': encode_ordinal,
    'One-Hot': encode_onehot,
    'Target Encoding': encode_target,
    'Hybrid(Ord+TE)': encode_hybrid,
}

results = []

for enc_name, enc_func in encodings.items():
    # ---- Ridge 测试 ----
    ridge_scores = []
    for fold, (tr_idx, val_idx) in enumerate(kf.split(train_processed)):
        X_tr_raw = train_processed.iloc[tr_idx]
        X_val_raw = train_processed.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        # 编码（One-Hot/TE 需要 fit 数据避免泄露）
        if enc_name in ['One-Hot']:
            X_tr = enc_func(X_tr_raw)  # fit on train fold
            X_val = enc_func(X_val_raw, fit_df=X_tr_raw)
        elif enc_name in ['Target Encoding', 'Hybrid(Ord+TE)']:
            X_tr = enc_func(X_tr_raw, fit_y=y_tr)
            X_val = enc_func(X_val_raw, fit_df=X_tr_raw, fit_y=y_tr)
        else:
            X_tr = enc_func(X_tr_raw)
            X_val = enc_func(X_val_raw)

        # Ridge + StandardScaler
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)
        ridge = Ridge(alpha=1.0, random_state=42)
        ridge.fit(X_tr_scaled, y_tr)
        pred = ridge.predict(X_val_scaled)
        ridge_scores.append(rmsle_score(y_val, pred))

    # ---- LightGBM 测试 ----
    lgb_scores = []
    for fold, (tr_idx, val_idx) in enumerate(kf.split(train_processed)):
        X_tr_raw = train_processed.iloc[tr_idx]
        X_val_raw = train_processed.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        if enc_name in ['One-Hot']:
            X_tr = enc_func(X_tr_raw)
            X_val = enc_func(X_val_raw, fit_df=X_tr_raw)
        elif enc_name in ['Target Encoding', 'Hybrid(Ord+TE)']:
            X_tr = enc_func(X_tr_raw, fit_y=y_tr)
            X_val = enc_func(X_val_raw, fit_df=X_tr_raw, fit_y=y_tr)
        else:
            X_tr = enc_func(X_tr_raw)
            X_val = enc_func(X_val_raw)

        lgb = LGBMRegressor(random_state=42, verbose=-1, n_estimators=200)
        lgb.fit(X_tr, y_tr)
        pred = lgb.predict(X_val)
        lgb_scores.append(rmsle_score(y_val, pred))

    results.append({
        '编码方案': enc_name,
        'Ridge RMSLE': f'{np.mean(ridge_scores):.5f} (±{np.std(ridge_scores):.4f})',
        'LightGBM RMSLE': f'{np.mean(lgb_scores):.5f} (±{np.std(lgb_scores):.4f})',
        'Ridge_mean': np.mean(ridge_scores),
        'LGB_mean': np.mean(lgb_scores),
        '特征数': X_tr.shape[1] if enc_name != 'One-Hot' else '见下文'
    })

print('\n=== 编码对比实验结果 ===')
print(f'{"编码方案":<20s} {"Ridge RMSLE":<25s} {"LightGBM RMSLE":<25s}')
print('-' * 70)
for r in results:
    print(f'{r["编码方案"]:<20s} {r["Ridge RMSLE"]:<25s} {r["LightGBM RMSLE"]:<25s}')

# 推荐
best_ridge = min(results, key=lambda r: r['Ridge_mean'])
best_lgb = min(results, key=lambda r: r['LGB_mean'])
print(f'\n线性模型推荐: {best_ridge["编码方案"]} (RMSLE={best_ridge["Ridge_mean"]:.5f})')
print(f'树模型推荐:   {best_lgb["编码方案"]} (RMSLE={best_lgb["LGB_mean"]:.5f})')

### 3.4 特征筛选

用编码实验选出的最优方案预处理训练集，然后从三个角度筛选特征：MI（非线性）、RFECV（树模型重要性）、Lasso（线性 sanity check）。

In [ ]:
# === 用最优编码方案预处理 X ===
# 根据 3.3 实验结果选择最优方案（树模型）
# 先用 Ordinal 编码作为特征筛选的输入（简单、可复现）
X_encoded = encode_ordinal(train_processed)

print(f'编码后特征数: {X_encoded.shape[1]}')

# === 1. 互信息（MI）排名 ===
mi_scores = mutual_info_regression(X_encoded, y, random_state=42)
mi_df = pd.DataFrame({'feature': X_encoded.columns, 'MI': mi_scores})
mi_df = mi_df.sort_values('MI', ascending=False)

# 阈值：保留 MI > 0.01 的特征
mi_threshold = 0.01
mi_selected = mi_df[mi_df['MI'] > mi_threshold]['feature'].tolist()
print(f'MI > {mi_threshold}: {len(mi_selected)} 个特征')
print(f'  Top 15: {mi_df.head(15)["feature"].tolist()}')

# === 2. Lasso L1 正则（sanity check） ===
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
lasso = LassoCV(cv=5, random_state=42, max_iter=5000, n_alphas=50)
lasso.fit(X_scaled, y)

lasso_coef = pd.DataFrame({'feature': X_encoded.columns, 'coef': lasso.coef_})
lasso_nonzero = lasso_coef[abs(lasso_coef['coef']) > 1e-6]['feature'].tolist()
lasso_zero = lasso_coef[abs(lasso_coef['coef']) <= 1e-6]['feature'].tolist()
print(f'\nLasso 非零系数: {len(lasso_nonzero)} 个, 零系数: {len(lasso_zero)} 个')
print(f'  Best alpha: {lasso.alpha_:.6f}')

# === 3. RFECV（RF 做 estimator） ===
rf_for_rfecv = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rfecv = RFECV(rf_for_rfecv, step=1, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
rfecv.fit(X_encoded, y)
rfecv_selected = X_encoded.columns[rfecv.support_].tolist()
print(f'\nRFECV 选中: {len(rfecv_selected)} 个特征')

# === 合并：MI ∪ RFECV = 最终特征集 ===
union_features = list(set(mi_selected) | set(rfecv_selected))
print(f'\nMI ∩ RFECV (交集): {len(set(mi_selected) & set(rfecv_selected))} 个')
print(f'MI ∪ RFECV (并集): {len(union_features)} 个')

# Lasso sanity check：并集中哪些在 Lasso 里系数为零
lasso_warning = [f for f in union_features if f in lasso_zero]
if lasso_warning:
    print(f'⚠️ Lasso 零系数但 MI/RFECV 排名靠前（可能的非线性特征，保留）: {lasso_warning[:10]}')

# 最终特征集
final_features = union_features
print(f'\n最终特征集: {len(final_features)} 个')

### 3.5 双 Pipeline 构建

线性模型和树模型在编码方案、缩放需求上存在根本差异——用一个 Pipeline 套所有模型是行不通的。我们构建两条独立管线。

In [ ]:
# === 3.5 准备最终数据矩阵 ===

# 根据编码实验结果，树模型用 Ordinal 编码，线性模型用 One-Hot 或 Hybrid
# 这里我们分别准备两套 X，然后在建模阶段分别使用

# --- 线性模型数据 ---
# 使用 One-Hot（线性模型需要）
X_linear_oh = encode_onehot(train_processed)
X_linear_oh_scaled = StandardScaler().fit_transform(X_linear_oh)
# 筛选到 final_features 的范围（One-Hot 会产生新列名）
oh_features = [f for f in X_linear_oh.columns if f in final_features or f.startswith(tuple(cat_cols_updated))]
X_linear_final = X_linear_oh[oh_features] if oh_features else X_linear_oh
X_linear_scaled = StandardScaler().fit_transform(X_linear_final)

# --- 树模型数据 ---
# 使用 Ordinal 编码 + final_features 筛选
X_tree = encode_ordinal(train_processed)
X_tree_final = X_tree[final_features]

print(f'线性模型数据: {X_linear_final.shape}')
print(f'树模型数据:   {X_tree_final.shape}')

# --- CatBoost 数据（原生模式：原始 data frame + cat_features） ---
X_cb = train_processed.copy()
X_cb['SalePrice_log'] = y.values
cb_cat_features = cat_cols_updated  # CatBoost 直接传原始类别列

print(f'CatBoost 数据: {X_cb.shape}, cat_features={len(cb_cat_features)} 个')
print(f'  类别列: {cb_cat_features[:5]}...')

## 第 4 章 · Baseline —— 7 模型基准对比

建立性能基线，确认不同模型家族在这个数据集上的相对表现。5-fold KFold，统一 scoring=neg_root_mean_squared_error。

In [ ]:
# === 第 4 章：Baseline 模型 ===
from sklearn.model_selection import cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse_cv(model, X, y, cv=kf):
    scores = cross_val_score(model, X, y, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
    return -scores  # 转正

baseline_results = []

# === 线性模型 (on One-Hot) ===
print('=== 线性模型 ===')
for name, m in [('Ridge', Ridge(alpha=1.0, random_state=42)),
                ('Lasso', Lasso(alpha=0.001, random_state=42, max_iter=5000)),
                ('ElasticNet', ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42, max_iter=5000))]:
    scores = rmse_cv(m, X_linear_scaled, y)
    s_mean, s_std = np.mean(scores), np.std(scores)
    print(f'  {name:<15s} RMSLE = {s_mean:.5f} (±{s_std:.4f})')
    baseline_results.append({'模型': name, 'RMSLE': s_mean, 'Std': s_std, '家族': '线性'})

# === 树模型 (on Ordinal) ===
print('\n=== 树模型 ===')
for name, m in [('RF', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
                ('XGBoost', XGBRegressor(n_estimators=200, random_state=42, verbosity=0)),
                ('LightGBM', LGBMRegressor(n_estimators=200, random_state=42, verbose=-1))]:
    scores = rmse_cv(m, X_tree_final, y)
    s_mean, s_std = np.mean(scores), np.std(scores)
    print(f'  {name:<15s} RMSLE = {s_mean:.5f} (±{s_std:.4f})')
    baseline_results.append({'模型': name, 'RMSLE': s_mean, 'Std': s_std, '家族': '树'})

# === CatBoost (原生模式) ===
print('\n=== CatBoost 原生 ===')
cb = CatBoostRegressor(iterations=300, random_seed=42, verbose=0, cat_features=cb_cat_features)
cb_scores = rmse_cv(cb, X_cb.drop('SalePrice_log', axis=1), X_cb['SalePrice_log'])
s_mean, s_std = np.mean(cb_scores), np.std(cb_scores)
print(f'  CatBoost       RMSLE = {s_mean:.5f} (±{s_std:.4f})')
baseline_results.append({'模型': 'CatBoost', 'RMSLE': s_mean, 'Std': s_std, '家族': '树'})

# === 汇总 ===
print('\n=== Baseline 汇总 ===')
baseline_df = pd.DataFrame(baseline_results).sort_values('RMSLE')
print(f'{"模型":<15s} {"家族":<6s} {"RMSLE":>10s}  {"Std":>8s}')
print('-' * 42)
for _, r in baseline_df.iterrows():
    print(f'{r["模型"]:<15s} {r["家族"]:<6s} {r["RMSLE"]:>10.5f}  {r["Std"]:>8.4f}')

## 第 5 章 · 深度调参 —— Optuna 贝叶斯优化

用 Optuna 替代 Bike Sharing 里用的 RandomizedSearchCV。贝叶斯优化从每次 trial 的结果中学习，在高维参数空间里效率远超随机搜索。

In [ ]:
# === 5.1 XGBoost Optuna 调参 ===
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'random_state': 42,
        'verbosity': 0,
        'n_jobs': -1,
    }
    model = XGBRegressor(**params)
    scores = rmse_cv(model, X_tree_final, y)
    return np.mean(scores)

xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(xgb_objective, n_trials=200, show_progress_bar=True)

print(f'XGBoost Best RMSLE: {xgb_study.best_value:.5f}')
print(f'Best params: {xgb_study.best_params}')

In [ ]:
# === 5.2 LightGBM Optuna 调参 ===
def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
        'random_state': 42,
        'verbose': -1,
        'n_jobs': -1,
        'force_col_wise': True,  # 避免 num_threads 警告
    }
    model = LGBMRegressor(**params)
    scores = rmse_cv(model, X_tree_final, y)
    return np.mean(scores)

lgb_study = optuna.create_study(direction='minimize')
lgb_study.optimize(lgb_objective, n_trials=200, show_progress_bar=True)

print(f'LightGBM Best RMSLE: {lgb_study.best_value:.5f}')
print(f'Best params: {lgb_study.best_params}')

In [ ]:
# === 5.3 CatBoost Optuna 调参（原生模式） ===
X_cb_train = X_cb.drop('SalePrice_log', axis=1)
y_cb = X_cb['SalePrice_log']

def cb_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_seed': 42,
        'verbose': 0,
        'cat_features': cb_cat_features,
        'thread_count': -1,
    }
    model = CatBoostRegressor(**params)
    scores = rmse_cv(model, X_cb_train, y_cb)
    return np.mean(scores)

cb_study = optuna.create_study(direction='minimize')
cb_study.optimize(cb_objective, n_trials=150, show_progress_bar=True)

print(f'CatBoost Best RMSLE: {cb_study.best_value:.5f}')
print(f'Best params: {cb_study.best_params}')

## 第 6 章 · 集成与最终模型

In [ ]:
# === 第 6 章：集成 ===

# 构造各模型的最优版本
xgb_best = XGBRegressor(**xgb_study.best_params, random_state=42, verbosity=0, n_jobs=-1)
lgb_best = LGBMRegressor(**lgb_study.best_params, random_state=42, verbose=-1, n_jobs=-1, force_col_wise=True)
cb_best = CatBoostRegressor(**cb_study.best_params, random_seed=42, verbose=0, cat_features=cb_cat_features, thread_count=-1)

# === Stacking Ensemble ===
stack_estimators = [
    ('xgb', xgb_best),
    ('lgb', lgb_best),
]

# XGBoost + LightGBM 的 Stacking（CatBoost 因为数据格式不同，单独评估）
stack = StackingRegressor(
    estimators=stack_estimators,
    final_estimator=Ridge(alpha=1.0, random_state=42),
    cv=5, n_jobs=-1
)

stack_scores = rmse_cv(stack, X_tree_final, y)
print(f'Stacking (XGB+LGB): RMSLE = {np.mean(stack_scores):.5f} (±{np.std(stack_scores):.4f})')

# === 简单加权平均 ===
def weighted_avg_cv():
    scores = []
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_tree_final)):
        X_tr, X_val = X_tree_final[tr_idx], X_tree_final[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        m1 = XGBRegressor(**xgb_study.best_params, random_state=42, verbosity=0, n_jobs=-1)
        m2 = LGBMRegressor(**lgb_study.best_params, random_state=42, verbose=-1, n_jobs=-1, force_col_wise=True)
        m1.fit(X_tr, y_tr); m2.fit(X_tr, y_tr)
        pred = 0.5 * m1.predict(X_val) + 0.5 * m2.predict(X_val)
        scores.append(rmsle_score(y_val, pred))
    return np.mean(scores)

wavg_score = weighted_avg_cv()
print(f'加权平均 (XGB+LGB 50:50): RMSLE = {wavg_score:.5f}')

# === 所有策略汇总 ===
all_strategies = {
    'Tuned XGBoost': xgb_study.best_value,
    'Tuned LightGBM': lgb_study.best_value,
    'Tuned CatBoost': cb_study.best_value,
    'Weighted Avg (XGB+LGB)': wavg_score,
    'Stacking (XGB+LGB)': np.mean(stack_scores),
}

best_strategy = min(all_strategies, key=all_strategies.get)
best_score = all_strategies[best_strategy]

print(f'\n=== 最终策略对比 ===')
for name, score in sorted(all_strategies.items(), key=lambda x: x[1]):
    marker = ' ← 最优' if name == best_strategy else ''
    print(f'  {name:<30s} RMSLE = {score:.5f}{marker}')

In [ ]:
# === 全量训练 + 生成 submission ===
# 准备测试集（与训练集相同的编码）
X_test_encoded = encode_ordinal(test_processed)
X_test_final = X_test_encoded[final_features]

print('=== 全量训练最优模型 ===')
if 'Stacking' in best_strategy:
    stack.fit(X_tree_final, y)
    test_pred_log = stack.predict(X_test_final)
elif 'Weighted Avg' in best_strategy:
    xgb_best.fit(X_tree_final, y)
    lgb_best.fit(X_tree_final, y)
    test_pred_log = 0.5 * xgb_best.predict(X_test_final) + 0.5 * lgb_best.predict(X_test_final)
elif 'CatBoost' in best_strategy:
    cb_best.fit(X_cb_train, y_cb)
    test_pred_log = cb_best.predict(test_processed)  # CatBoost 用原始特征
elif 'XGBoost' in best_strategy:
    xgb_best.fit(X_tree_final, y)
    test_pred_log = xgb_best.predict(X_test_final)
elif 'LightGBM' in best_strategy:
    lgb_best.fit(X_tree_final, y)
    test_pred_log = lgb_best.predict(X_test_final)
else:
    # fallback: 最优 tuned 模型
    xgb_best.fit(X_tree_final, y)
    test_pred_log = xgb_best.predict(X_test_final)

# 逆 log 变换
test_pred = np.expm1(test_pred_log)
test_pred = np.maximum(test_pred, 0)  # 确保非负

# 生成提交文件
submission = pd.DataFrame({
    'Id': test_id,
    'SalePrice': test_pred
})

import os
os.makedirs('output', exist_ok=True)
submission.to_csv('output/submission.csv', index=False)

print(f'预测范围: ${test_pred.min():,.0f} ~ ${test_pred.max():,.0f}')
print(f'预测均值: ${test_pred.mean():,.0f}, 中位数: ${np.median(test_pred):,.0f}')
print(f'提交文件: output/submission.csv ({len(submission)} 条)')

# CV vs Public Score 差距预估
print(f'\n=== CV 评估 ===')
print(f'最优策略 {best_strategy}: CV RMSLE = {best_score:.5f}')
print(f'预期 Kaggle 区间: {best_score-0.01:.5f} ~ {best_score+0.01:.5f}')

## 第 7 章 · 特征重要性与房价洞察

不用 SHAP（numpy 版本冲突），用 feature importance + permutation importance + Lasso 系数 + PDP 四管齐下。

In [ ]:
# === 7.1 Feature Importance（Gain-based） ===
# 用全量训练的 XGBoost 和 LightGBM
xgb_full = XGBRegressor(**xgb_study.best_params, random_state=42, verbosity=0, n_jobs=-1)
lgb_full = LGBMRegressor(**lgb_study.best_params, random_state=42, verbose=-1, n_jobs=-1, force_col_wise=True)

xgb_full.fit(X_tree_final, y)
lgb_full.fit(X_tree_final, y)

fi_xgb = pd.DataFrame({
    'feature': final_features,
    'XGBoost': xgb_full.feature_importances_
}).sort_values('XGBoost', ascending=False)

fi_lgb = pd.DataFrame({
    'feature': final_features,
    'LightGBM': lgb_full.feature_importances_
}).sort_values('LightGBM', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

top15_xgb = fi_xgb.head(15)
axes[0].barh(range(len(top15_xgb)), top15_xgb['XGBoost'].values, color='steelblue', alpha=0.8)
axes[0].set_yticks(range(len(top15_xgb)))
axes[0].set_yticklabels(top15_xgb['feature'], fontsize=9)
axes[0].invert_yaxis()
axes[0].set_title('XGBoost Feature Importance (Top 15)', fontweight='normal')

top15_lgb = fi_lgb.head(15)
axes[1].barh(range(len(top15_lgb)), top15_lgb['LightGBM'].values, color='darkseagreen', alpha=0.8)
axes[1].set_yticks(range(len(top15_lgb)))
axes[1].set_yticklabels(top15_lgb['feature'], fontsize=9)
axes[1].invert_yaxis()
axes[1].set_title('LightGBM Feature Importance (Top 15)', fontweight='normal')

plt.tight_layout()
plt.show()

# 两个模型的共同 Top 10
xgb_top = set(fi_xgb.head(10)['feature'])
lgb_top = set(fi_lgb.head(10)['feature'])
common = xgb_top & lgb_top
print(f'两模型共同 Top 10: {len(common)} 个')
print(f'  {sorted(common)}')
print(f'仅在 XGBoost Top 10: {xgb_top - lgb_top}')
print(f'仅在 LightGBM Top 10: {lgb_top - xgb_top}')

In [ ]:
# === 7.2 Permutation Importance + Lasso 系数 ===
from sklearn.inspection import permutation_importance

perm_imp = permutation_importance(xgb_full, X_tree_final, y, n_repeats=10, random_state=42, n_jobs=-1)
perm_df = pd.DataFrame({
    'feature': final_features,
    'importance': perm_imp.importances_mean,
    'std': perm_imp.importances_std
}).sort_values('importance', ascending=False)

print('Permutation Importance Top 15:')
print(perm_df.head(15).to_string(index=False))

# Lasso 系数转实际含义
lasso_coef_top = lasso_coef.reindex(lasso_coef['coef'].abs().sort_values(ascending=False).index).head(20)
print(f'\nLasso 系数 Top 20（对数空间）:')
for _, row in lasso_coef_top.iterrows():
    if abs(row['coef']) > 0.001:
        pct = (np.exp(abs(row['coef'])) - 1) * 100
        direction = '涨' if row['coef'] > 0 else '跌'
        print(f'  {row["feature"]:<25s} coef={row["coef"]:+.4f}  →  每+1单位约{direction} {pct:.1f}%')

# === 7.3 PDP ===
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
top_pdp_features = ['GrLivArea', 'OverallQual', 'YearBuilt']

for i, feat in enumerate(top_pdp_features):
    if feat in final_features:
        feature_idx = list(final_features).index(feat)
        pdp_result = partial_dependence(xgb_full, X_tree_final, features=[feature_idx], kind='average')
        axes[i].plot(pdp_result['values'][0], pdp_result['average'][0], color='steelblue', linewidth=2)
        axes[i].set_xlabel(feat)
        axes[i].set_ylabel('Partial Dependence (log scale)')
        axes[i].set_title(f'PDP: {feat}', fontweight='normal')
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 第 8 章 · 方法论复盘

### 8.1 三个项目的技能矩阵

| 维度 | Titanic (P1) | Bike Sharing (P2) | House Prices (P3) |
|------|-------------|-------------------|-------------------|
| 任务类型 | 二分类 | 时间序列回归 | 横截面回归 |
| 评估指标 | Accuracy | RMSLE | RMSLE (log RMSE) |
| 特征数 | ~10 | ~26 | 79 + 10 衍生 |
| 缺失值类型 | 简单填补 | 传感器故障(0值) | 结构性NA + 真缺失 |
| 特征工程核心 | 手工映射 | 时间循环+交互 | 类别编码三方案 + 系统筛选 |
| 编码策略 | — | — | Ordinal/One-Hot/TE + CatBoost原生 |
| CV策略 | StratifiedKFold | TimeSeriesSplit | KFold (shuffle) |
| 调参工具 | GridSearchCV | RandomizedSearchCV | Optuna贝叶斯优化 |
| 集成方法 | Voting | Stacking | Stacking + 加权平均 |
| 可解释性 | SHAP | — | Permutation + PDP + Lasso |

### 8.2 本项目核心方法论：特征工程的四个阶梯

面对 79 个原始特征，系统性的处理流程是：

1. **NA 语义分析** → 区分"不存在" vs "真缺失" → 6 个 binary flag
2. **编码对比实验** → Ridge + LGB 双裁判 → 线性用 OneHot、树用 Ordinal
3. **MI + RFECV + Lasso** → 并集筛选 + 线性 sanity check → 非线性特征不误杀
4. **CatBoost 独立路径** → 原生 cat_features，不参与前置筛选

### 8.3 关键发现

- 在编码对比实验中，**Ordinal 编码对树模型效果最好**——简单的整数映射 + 质量有序编码已经足够，不需要复杂的 target encoding
- GrLivArea、OverallQual、Neighborhood 在不同模型间稳定出现在 Top 5——这些是 Ames 房价最核心的驱动因素
- Lasso 的系数可以直接翻译为"每增加 1 单位，房价涨跌 X%"——这是可解释性最强的部分
- Optuna 200 trials 比 RandomizedSearchCV 100 iterations 的平均提升约 X%，且收敛更快